# Lab 20 — Multi-Agent Research Demo Notebook

Notebook prototype cho bài lab. **Tất cả TODO đã được hoàn thành** — notebook chạy
end-to-end với mock services, không cần API key.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> Logic chính thức đã nằm trong `src/multi_agent_research_lab/` (xem bảng mapping ở cuối).
> Notebook giữ lại để prototype nhanh và giải thích từng bước.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [1]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.errors import StudentTodoError
from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

✅ Import OK — package sẵn sàng


## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [2]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

Iteration: 1
Route history: ['researcher']
Trace: [{'name': 'demo', 'payload': {'note': 'first route recorded'}}]


## 2. Mock Services

Mock để demo chạy không cần API key. Bản chính thức trong `src/services/` nối
provider thật (OpenAI) và corpus offline.

- `MockSearchClient`: nguồn giả lập cố định.
- `MockLLMClient`: **đã hoàn thành** — trích citation id từ prompt và echo lại
  theo đúng shape, đủ để pipeline và citation checker chạy thật.

In [3]:
import re
from dataclasses import dataclass

from multi_agent_research_lab.services.llm_client import LLMResponse


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            metadata={"source_id": "rag_guide"},
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            metadata={"source_id": "rag_survey"},
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            metadata={"source_id": "ft_when"},
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None

class MockLLMClient:
    """SOLVED: deterministic stand-in — cùng interface với LLMClient."""

    def __init__(self, model: str = 'mock-model') -> None:
        self.model = model
        self.call_count = 0
        self.total_cost_usd = 0.0

    def complete(self, system_prompt: str, user_prompt: str) -> LLMResponse:
        self.call_count += 1
        pattern = r'\[([A-Za-z0-9_\\-]{1,40})\]'
        cites = list(dict.fromkeys(re.findall(pattern, user_prompt)))
        lines = [f'- Điểm {i} dựa trên [{c}]' for i, c in enumerate(cites, 1)]
        body = '\n'.join(lines) or '- Không có evidence.'
        content = '# Trả lời (mock)\n\n' + body
        self.total_cost_usd += 0.0001
        return LLMResponse(
            content=content,
            input_tokens=len(user_prompt) // 4,
            output_tokens=len(content) // 4,
            cost_usd=0.0001,
        )


llm = MockLLMClient()
print('MockLLMClient OK:', llm.complete('You are a writer.', 'Test [a1]').content[:50])

MockLLMClient OK: # Trả lời (mock)

- Điểm 1 dựa trên [a1]


## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: gọi search, ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: **đã hoàn thành** — tổng hợp `sources` thành `analysis_notes`.
- `DemoWriterAgent`: **đã hoàn thành** — viết `final_answer` kèm citation từ allow-list.

In [4]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state

class DemoAnalystAgent:
    """SOLVED: tổng hợp sources thành analysis_notes."""

    name = 'analyst'

    def __init__(self, llm_client) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        evidence = '\n'.join(
            f"[{d.metadata.get('source_id', d.title)}] {d.snippet}"
            for d in state.sources
        )
        response = self.llm_client.complete(
            'You are an analyst. Structure the evidence into key claims.',
            f'Question: {state.request.query}\n\nEvidence:\n{evidence}',
        )
        state.analysis_notes = response.content
        state.agent_results.append(
            AgentResult(
                agent=AgentName.ANALYST,
                content=response.content,
                metadata={'num_sources': len(state.sources)},
            )
        )
        state.add_trace_event('analyst.done', {'chars': len(response.content)})
        return state


class DemoWriterAgent:
    """SOLVED: viết final_answer kèm citation."""

    name = 'writer'

    def __init__(self, llm_client) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        ids = [str(d.metadata.get('source_id', d.title)) for d in state.sources]
        allowed = ', '.join('[' + i + ']' for i in ids)
        response = self.llm_client.complete(
            'You are a writer. Cite only the allowed ids.',
            f'Question: {state.request.query}\n\n'
            f'Analysis:\n{state.analysis_notes}\n\n'
            f'Allowed citation ids: {allowed}',
        )
        state.final_answer = response.content
        state.agent_results.append(
            AgentResult(agent=AgentName.WRITER, content=response.content, metadata={})
        )
        state.add_trace_event('writer.done', {'chars': len(response.content)})
        return state


print('Demo agents OK')

Demo agents OK


## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [5]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """SOLVED: trả về một trong 'researcher' | 'analyst' | 'writer' | 'done'."""
    # Guard chống vòng lặp vô hạn — GIỮ NGUYÊN dòng này
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    # Route theo field còn thiếu, đúng thứ tự phụ thuộc.
    if not state.sources:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"


print("✅ Routing policy sẵn sàng")


✅ Routing policy sẵn sàng


## 5. Mini Workflow Loop

Vòng lặp điều phối **đã viết sẵn** — chỉ chạy được sau khi bạn hoàn thành các TODO ở trên. Đây chính là logic bạn sẽ chuyển thành LangGraph nodes/edges trong `graph/workflow.py`.

In [6]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


try:
    final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
    print("Route history:", final_state.route_history)
    print("\n=== FINAL ANSWER ===\n")
    print(final_state.final_answer)
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")
    print("→ Quay lại các ô trên, implement xong rồi chạy lại ô này.")

Route history: ['researcher', 'analyst', 'writer', 'done']

=== FINAL ANSWER ===

# Trả lời (mock)

- Điểm 1 dựa trên [rag_guide]
- Điểm 2 dựa trên [rag_survey]
- Điểm 3 dựa trên [ft_when]


## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh. Baseline single-agent (1 lần gọi LLM, không search) **bạn tự viết**.

In [7]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """SOLVED baseline: một lần gọi LLM duy nhất, không search."""
    state = ResearchState(request=ResearchQuery(query=query_text, max_sources=3))
    state.record_route('single_agent')
    response = MockLLMClient().complete(
        'You are a research assistant.', state.request.query
    )
    state.final_answer = response.content
    return state


def compute_citation_coverage(state: ResearchState) -> float:
    """SOLVED: tỷ lệ nguồn được nhắc đến trong final_answer."""
    if not state.sources or not state.final_answer:
        return 0.0
    valid = {str(d.metadata.get('source_id', d.title)) for d in state.sources}
    pattern = r'\[([A-Za-z0-9_\\-]{1,40})\]'
    cited = set(re.findall(pattern, state.final_answer)) & valid
    return len(cited) / len(valid)


demo_query = 'So sánh RAG và fine-tuning cho domain adaptation'

state_single, m_single = run_benchmark('baseline', demo_query, run_single_agent)
state_multi, m_multi = run_benchmark('multi-agent', demo_query, run_demo_workflow)

cov_s = compute_citation_coverage(state_single)
cov_m = compute_citation_coverage(state_multi)
print(f'baseline   : {m_single.latency_seconds:.3f}s  coverage={cov_s:.0%}')
print(f'multi-agent: {m_multi.latency_seconds:.3f}s  coverage={cov_m:.0%}')

baseline   : 0.000s  coverage=0%
multi-agent: 0.000s  coverage=100%


## 7. Next Steps — chuyển sang `src/`

Khi notebook chạy end-to-end, chuyển logic vào code chính thức:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` |
| `MockSearchClient` → provider thật | `services/search_client.py` |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py` |

Sau đó verify:
```bash
make lint && make test
python -m multi_agent_research_lab.cli run --query "..."
bash scripts/check_todos.sh   # đảm bảo không còn TODO trong src/
```